# 📘 The AI Engineer's LLM Workbook

**14 Chapters · 14 Google Colab Notebooks · Beginner to Production**

---

*© 2026 JAWNVION LLC — www.jawnvion.com — peter@jawnvion.com*

*Licensed for individual use. Do not redistribute.*

---

## What's Inside

| # | Chapter |
|---|---------|
| 01 | AI Fundamentals & Problem Framing |
| 02 | Data Science Toolkit (NumPy, Pandas, Matplotlib) |
| 03 | Neural Networks from Scratch |
| 04 | Transformers Architecture Deep Dive |
| 05 | HuggingFace & Pre-Trained Models |
| 06 | QLoRA Fine-Tuning |
| 07 | DPO Alignment Training |
| 08 | Retrieval-Augmented Generation (RAG) |
| 09 | Model Evaluation & Benchmarking |
| 10 | FastAPI Deployment |
| 11 | Monitoring & Observability |
| 12 | Security for AI Systems |
| 13 | Cost Optimization & Quantization |
| 14 | Capstone: End-to-End LLM Project |

---

> **How to use:** Click **Runtime → Run All** in Google Colab, or run cells one at a time.
> Each chapter builds on the last — complete them in order for best results.

---


# Chapter 6: Fine-Tuning with QLoRA
**JAWNVION LLC** | peter@jawnvion.com

Fine-tune TinyLlama-1.1B on 1,000 Alpaca instruction examples using 4-bit QLoRA on a free T4 GPU.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# ── Cell 1: GPU Check ──────────────────────────────────────────
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✓  GPU : {name}')
    print(f'   VRAM: {total:.1f} GB')
else:
    raise RuntimeError('No GPU found. Change runtime type to T4 GPU.')


In [ ]:
# ── Cell 2: Install Dependencies ───────────────────────────────
# Pin tokenizers first so it installs from a pre-built wheel (avoids Rust compile)
!pip install -q "tokenizers>=0.22,<0.24"
!pip install -q -U transformers trl peft bitsandbytes accelerate datasets
print('✓  Packages installed — if prompted to restart runtime, do so then Run All again')


In [ ]:
# ── Cell 3: Imports ─────────────────────────────────────────────
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

print('✓  All imports successful')


In [ ]:
# ── Cell 4: Configuration ───────────────────────────────────────
MODEL_NAME        = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
OUTPUT_DIR        = '/content/tinyllama-qlora'
NUM_TRAIN_EXAMPLES = 1000
MAX_SEQ_LEN       = 512

print(f'Model : {MODEL_NAME}')
print(f'Output: {OUTPUT_DIR}')


In [ ]:
# ── Cell 5: Load Model in 4-bit (QLoRA) ────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Loading model in 4-bit...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
)
model.config.use_cache = False

vram = torch.cuda.memory_allocated() / 1e9
print(f'✓  Model loaded  |  VRAM used: {vram:.2f} GB')


In [ ]:
# ── Cell 6: LoRA Adapter Configuration ─────────────────────────
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# ── Cell 7: Load & Format Dataset ──────────────────────────────
raw = load_dataset('tatsu-lab/alpaca', split=f'train[:{NUM_TRAIN_EXAMPLES}]')

def format_example(ex):
    if ex['input']:
        prompt = (
            f"### Instruction:\n{ex['instruction']}\n\n"
            f"### Input:\n{ex['input']}\n\n"
            f"### Response:\n{ex['output']}"
        )
    else:
        prompt = (
            f"### Instruction:\n{ex['instruction']}\n\n"
            f"### Response:\n{ex['output']}"
        )
    return {'text': prompt}

dataset = raw.map(format_example, remove_columns=raw.column_names)
print(f'✓  Dataset ready: {len(dataset)} examples')
print('Sample:')
print(dataset[0]['text'][:300])


In [ ]:
# ── Cell 8: Training Configuration ─────────────────────────────
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim='paged_adamw_8bit',
    learning_rate=2e-4,
    weight_decay=0.001,
    lr_scheduler_type='cosine',
    warmup_steps=6,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    dataset_text_field='text',
    packing=False,
    logging_steps=25,
    save_steps=100,
    save_total_limit=2,
    report_to='none',
)

total_steps = (NUM_TRAIN_EXAMPLES // (4 * 4)) * 2
print('✓  SFTConfig ready')
print(f'   Effective batch : {4*4}')
print(f'   ~Total steps    : {total_steps}')
print(f'   Warmup steps    : {sft_config.warmup_steps}')


In [ ]:
# ── Cell 9: Build Trainer ───────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config,
    processing_class=tokenizer,
)

print('✓  SFTTrainer created and ready.')

if torch.cuda.is_available():
    used     = torch.cuda.memory_allocated() / 1e9
    total    = torch.cuda.get_device_properties(0).total_memory / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f'   VRAM allocated : {used:.2f} GB')
    print(f'   VRAM headroom  : {total - reserved:.1f} GB')
    if total - reserved < 2.0:
        print('△  Low VRAM — reduce batch_size to 2 if OOM')
    else:
        print('✓  Sufficient VRAM headroom')


In [ ]:
# ── Cell 10: Train ──────────────────────────────────────────────
print('=' * 60)
print('  Starting QLoRA Fine-Tuning')
print(f'  Model  : {MODEL_NAME}')
print(f'  Data   : {NUM_TRAIN_EXAMPLES} Alpaca examples')
print(f'  Epochs : 2')
print('=' * 60)
print('Training loss will print every 25 steps.')
print('A decreasing loss means the model is learning.')
print()

start_time = time.time()
train_result = trainer.train()
elapsed = time.time() - start_time

print()
print(f'✓  Training complete in {elapsed/60:.1f} min')
print(f'   Final loss: {train_result.training_loss:.4f}')


In [ ]:
# ── Cell 11: Save Adapter ───────────────────────────────────────
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'✓  Adapter saved to {OUTPUT_DIR}')


In [ ]:
# ── Cell 12: Before / After Inference Comparison ───────────────
from peft import PeftModel
from transformers import pipeline

TEST_PROMPT = (
    '### Instruction:\n'
    'Explain what machine learning is in one sentence.\n\n'
    '### Response:\n'
)

def generate(mdl, prompt, max_new=120):
    inputs = tokenizer(prompt, return_tensors='pt').to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens=max_new,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                            skip_special_tokens=True)

print('FINE-TUNED RESPONSE:')
print(generate(trainer.model, TEST_PROMPT))


## ✓ Chapter 6 Complete

You have:
- Loaded TinyLlama-1.1B in 4-bit NF4 quantization
- Applied LoRA adapters (r=16, 7 target modules)
- Fine-tuned on 1,000 Alpaca instruction examples
- Saved the adapter weights to `/content/tinyllama-qlora`

**Next:** Chapter 7 — RLHF & DPO alignment training